# Generación de instancias RDF a partir de archivos CSV

En este notebook veremos cómo transformar datos en archivos CSV en un grafo RDF usando `rdflib`.

La idea central es sencilla:

- cada **fila** del CSV puede representar una instancia
- una columna puede aportar el identificador de esa instancia
- otras columnas pueden convertirse en **literales**
- algunas columnas pueden representar relaciones con **otros recursos RDF**

El ejemplo utiliza películas y directores. Después encontrará un espacio para adaptar el procedimiento a su modelo de Hogwarts.


## 1. Preparación

Primero importamos las bibliotecas necesarias y definimos dos espacios de nombres:

- `VOC`: para clases y propiedades del vocabulario
- `RES`: para las instancias o recursos concretos

Esta separación sigue la idea de mantener distintos los IRI del vocabulario y los IRI de los datos.


In [28]:
!pip install rdflib

In [29]:
import csv
from pathlib import Path

from rdflib import Graph, Namespace, Literal, RDF, XSD

VOC = Namespace("http://example.org/cine/vocab/")
RES = Namespace("http://example.org/cine/resource/")

g = Graph()
g.bind("cine", VOC)
g.bind("res", RES)


## 2. Archivos CSV del ejemplo

Usaremos dos archivos:

### `directores.csv`

```csv
id,nombre
d001,Bong Joon-ho
d002,Denis Villeneuve
d003,Hayao Miyazaki
```

### `peliculas.csv`

```csv
id,titulo,anio,director_id
p001,Parasite,2019,d001
p002,Arrival,2016,d002
p003,Spirited Away,2001,d003
```

Observe que `peliculas.csv` no guarda el nombre del director como texto. En su lugar utiliza `director_id`.

Esto permite representar al director como un **recurso RDF** y no simplemente como un literal.


In [30]:
# Crear los CSV del ejemplo para poder ejecutar el cuaderno directamente (quedan guardados en el apartado "Archivos").

Path("directores.csv").write_text(
    "id,nombre\n"
    "d001,Bong Joon-ho\n"
    "d002,Denis Villeneuve\n"
    "d003,Hayao Miyazaki\n",
    encoding="utf-8"
)

Path("peliculas.csv").write_text(
    "id,titulo,anio,director_id\n"
    "p001,Parasite,2019,d001\n"
    "p002,Arrival,2016,d002\n"
    "p003,Spirited Away,2001,d003\n",
    encoding="utf-8"
)

print("Archivos creados.")


Archivos creados.


## 3. Cargar los directores

Cada fila de `directores.csv` representa una instancia de la clase `Director`.

Para cada fila:

1. construimos un IRI a partir de la columna `id`
2. declaramos que el recurso es un `cine:Director`
3. agregamos su nombre como literal


In [31]:
with open("directores.csv", encoding="utf-8") as archivo:
    for fila in csv.DictReader(archivo):
        director = RES[fila["id"]]

        g.add((director, RDF.type, VOC.Director))
        g.add((director, VOC.nombre, Literal(fila["nombre"])))


## 4. Cargar las películas

Cada fila de `peliculas.csv` representa una instancia de la clase `Pelicula`.

En este caso hay tres tipos de transformación:

- `titulo` se convierte en un literal de texto
- `anio` se convierte en un literal con tipo `xsd:integer`
- `director_id` se convierte en un IRI que apunta a una instancia de `Director`

La última transformación es especialmente importante: RDF permite relacionar recursos entre sí, no solo almacenar texto.


In [32]:
with open("peliculas.csv", encoding="utf-8") as archivo:
    for fila in csv.DictReader(archivo):
        pelicula = RES[fila["id"]]
        director = RES[fila["director_id"]]

        g.add((pelicula, RDF.type, VOC.Pelicula))
        g.add((pelicula, VOC.titulo, Literal(fila["titulo"])))
        g.add((pelicula, VOC.anio,
               Literal(fila["anio"], datatype=XSD.integer)))
        g.add((pelicula, VOC.dirigidaPor, director))


## 5. Revisar el grafo generado

Serializamos el grafo en Turtle para comprobar las instancias y relaciones creadas.

Compare especialmente estas dos posibilidades:

```turtle
res:p001 cine:dirigidaPor "Bong Joon-ho" .
```

y

```turtle
res:p001 cine:dirigdaPor res:d001 .
```

En el segundo caso, el director tiene identidad propia como recurso RDF y puede tener sus propias propiedades.


In [33]:
print(g.serialize(format="turtle"))


@prefix cine: <http://example.org/cine/vocab/> .
@prefix res: <http://example.org/cine/resource/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

res:p001 a cine:Pelicula ;
    cine:anio 2019 ;
    cine:dirigidaPor res:d001 ;
    cine:titulo "Parasite" .

res:p002 a cine:Pelicula ;
    cine:anio 2016 ;
    cine:dirigidaPor res:d002 ;
    cine:titulo "Arrival" .

res:p003 a cine:Pelicula ;
    cine:anio 2001 ;
    cine:dirigidaPor res:d003 ;
    cine:titulo "Spirited Away" .

res:d001 a cine:Director ;
    cine:nombre "Bong Joon-ho" .

res:d002 a cine:Director ;
    cine:nombre "Denis Villeneuve" .

res:d003 a cine:Director ;
    cine:nombre "Hayao Miyazaki" .




## 6. Algunas comprobaciones

Antes de adaptar el ejercicio, revise:

- ¿cuántas tripletas tiene el grafo?
- ¿cuántas instancias de `Pelicula` se generaron?
- ¿cuántas instancias de `Director` se generaron?
- ¿qué columnas del CSV se transformaron en literales?
- ¿qué columna produjo una relación entre dos recursos


In [34]:
print("Número total de tripletas:", len(g))

peliculas = list(g.subjects(RDF.type, VOC.Pelicula))
directores = list(g.subjects(RDF.type, VOC.Director))

print("Películas:", len(peliculas))
print("Directores:", len(directores))


Número total de tripletas: 18
Películas: 3
Directores: 3


# 7. Su turno: adapte el procedimiento a su modelo de Hogwarts

Utilice ahora el modelo RDFS que construyó en el ejercicio anterior.

Debe:

1. diseñar uno o más archivos CSV para representar las instancias de su modelo;
2. incluir al menos **3 instancias de cada clase** requerida en el ejercicio;
3. identificar qué columnas representan literales y cuáles representan relaciones con otros recursos;
4. respetar el esquema de IRI que diseñó y documentó previamente;
5. asignar tipos de datos adecuados cuando corresponda;
6. generar las tripletas con `rdflib`;
7. serializar el resultado en Turtle y verificar que el grafo corresponde con su modelo.

## Antes de programar

Complete estas decisiones:

- ¿Qué representa cada fila de cada CSV?<br>
El de profesores.csv cada fila representa un recurso es decir, un profesor.
- ¿Qué columna utilizará para construir el IRI de cada recurso?<br>
Utilizaré las columnas asignatura y profesor_id para construir el IRI completo
- ¿Qué columnas serán literales?<br>
La columna asignatura ("semantic", "paradigmas", "imagenes")
- ¿Qué columnas harán referencia a otras instancias?<br>
profesor_id hace referencia al Profesor
casa_id hace referencia a la Casa
- ¿Qué tipos de datos necesita?<br>
Necesito string para los nombres de las asignaturas y los nombres de profesores


### Diseño de sus archivos CSV

Describa aquí los CSV que utilizará.

**CSV 1**

- Nombre del archivo: profesores.csv
- Cada fila representa: un profesor
- Columnas:id, nombre
- Clase RDF asociada: Persona

**CSV 2**

- Nombre del archivo: asignaturas.csv
- Cada fila representa: una asignatura
- Columnas:id, asignatura, profesor_id
- Clase RDF asociada: Asignatura

**CSV 3**

- Nombre del archivo: casas.csv
- Cada fila representa: una casa
- Columnas:id, nombre
- Clase RDF asociada: Casa

**CSV 4**

- Nombre del archivo: casas_profesores.csv
- Cada fila representa: la casa a la que pertenece un profesor
- Columnas:id, casa_id, profesor_id
- Clase RDF asociada: ex:Casa y ex:Profesor

Agregue más archivos si su modelo los necesita.


# Ejercicio 5.1 Diseño del modelos RDFS

## Clases
ex: Casa rdf:type rdfs: Class .
ex: Asignatura rdf:type rdfs: Class .
ex: Persona rdf:type rdfs: Class .

## Subclases
ex: Estudiante rdf:subClassOf rdfs: Persona .
ex: Profesor rdf:subClassOf rdfs: Persona .

## Propiedades
ex: perteneceACasa
ex: enseñaAsignatura
ex: realizoAsignatura
ex: realizoHechizoA
ex: dictadaPor

## Subpropiedad
ex: aproboAsignatura rdf:subPropertyOf ex:realizoAsignatura .
ex: reproboAsignatura rdf:subPropertyOf ex:realizoAsignatura .

## Dominio y rango
:perteneceACasa rdf:type rdf:Property .
:perteneceACasa rdfs:domain rdf: Persona .
:perteneceACasa rdfs:range rdf: Casa .

:enseñaAsignatura rdf:type rdf:Property .
:enseñaAsignatura rdfs:domain rdf: Persona .
:enseñaAsignatura rdfs:range rdf: Asignatura .

:dictadaPor rdf:type rdf:Property .
:dictadaPor rdf:domain rdf:Asignatura .
:dictadaPor rdf:range rdf:Persona .


## Antes de crear el modelo: Esquema de IRI
prefix : <http://www.hogwarts.org/vocab#>
prefix ex: <http://www.hogwarts.org>
prefix rdf: <http://www.hogwarts.org/1999/02/22-rdf-syntax-ns#>
prefix rdfs: <http://www.hogwarts.org/2000/01/rdf-schema#>
prefix xsd: <http://www.hogwarts.org/2001/XMLSchema#>



# Ejercicio 5.2

In [35]:
# TODO: defina aquí los Namespace de su modelo de Hogwarts.

VOC_H = Namespace("http://hogwarts.org/hog/vocab/")
RES_H = Namespace("http://hogwarts.org/hog/resource/")

g_hogwarts = Graph()
g_hogwarts.bind("hog", VOC_H)
g_hogwarts.bind("res", RES_H)




In [36]:
# TODO: cargue aquí su primer archivo CSV y genere las tripletas correspondientes.
Path("profesores.csv").write_text(
    "id,nombre\n"
    "p001,ruben\n"
    "p002,nicolas\n"
    "p003,juan pablo\n",
    encoding="utf-8"
)



50

In [37]:
# TODO: cargue aquí los demás archivos CSV que necesite.

Path("asignaturas.csv").write_text(
    "id,asignatura,profesor_id\n"
    "a001,semantic,p001\n"
    "a002,paradigmas,p002\n"
    "a003,imagenes,p003\n",
    encoding="utf-8"
)

# Para las casas
Path("casas.csv").write_text(
    "id,nombre\n"
    "c001,Gryffindor\n"
    "c002,Slytherin\n"
    "c003,Ravenclaw\n"
    "c004,Hufflepuff\n",
    encoding="utf-8"
)

# Asignacion de casas
Path("casas_profesores.csv").write_text(
    "id,casa_id,profesor_id\n"
    "cp001,c001,p001\n"
    "cp002,c002,p002\n"
    "cp003,c003,p003\n",
    encoding="utf-8"
)




71

Cargar los profesores

In [38]:
with open("profesores.csv", encoding="utf-8") as archivo:
    for fila in csv.DictReader(archivo):
        profesor = RES_H[fila["id"]]

        g_hogwarts.add((profesor, RDF.type, VOC_H.Profesor))
        g_hogwarts.add((profesor, VOC_H.nombre, Literal(fila["nombre"])))

Cargar las casas

In [39]:
with open("casas.csv", encoding="utf-8") as archivo:
    for fila in csv.DictReader(archivo):
        casa = RES_H[fila["id"]]

        g_hogwarts.add((casa, RDF.type, VOC_H.Casa))
        g_hogwarts.add((casa, VOC_H.nombre, Literal(fila["nombre"])))

Cargar las tripletas de asignaturas dictadas por profesores

In [40]:
with open("asignaturas.csv", encoding="utf-8") as archivo:
    for fila in csv.DictReader(archivo):
        asignatura = RES_H[fila["id"]]
        profesor = RES_H[fila["profesor_id"]]

        g_hogwarts.add((asignatura, RDF.type, VOC_H.Asignatura))
        g_hogwarts.add((asignatura, VOC_H.asignatura, Literal(fila["asignatura"])))
        g_hogwarts.add((asignatura, VOC_H.dictadaPor, profesor))

Cargar las tripletas de casas a los profesores

In [41]:
with open("casas_profesores.csv", encoding="utf-8") as archivo:
    for fila in csv.DictReader(archivo):
        casa = RES_H[fila["casa_id"]]
        profesor = RES_H[fila["profesor_id"]]

        g_hogwarts.add((casa, RDF.type, VOC_H.Casa))
        g_hogwarts.add((profesor, RDF.type, VOC_H.Profesor))
        g_hogwarts.add((profesor, VOC_H.perteneceACasa, casa))


In [42]:
# TODO: serialice y revise el grafo final.

print(g_hogwarts.serialize(format="turtle"))


@prefix hog: <http://hogwarts.org/hog/vocab/> .
@prefix res: <http://hogwarts.org/hog/resource/> .

res:a001 a hog:Asignatura ;
    hog:asignatura "semantic" ;
    hog:dictadaPor res:p001 .

res:a002 a hog:Asignatura ;
    hog:asignatura "paradigmas" ;
    hog:dictadaPor res:p002 .

res:a003 a hog:Asignatura ;
    hog:asignatura "imagenes" ;
    hog:dictadaPor res:p003 .

res:c004 a hog:Casa ;
    hog:nombre "Hufflepuff" .

res:c001 a hog:Casa ;
    hog:nombre "Gryffindor" .

res:c002 a hog:Casa ;
    hog:nombre "Slytherin" .

res:c003 a hog:Casa ;
    hog:nombre "Ravenclaw" .

res:p001 a hog:Profesor ;
    hog:nombre "ruben" ;
    hog:perteneceACasa res:c001 .

res:p002 a hog:Profesor ;
    hog:nombre "nicolas" ;
    hog:perteneceACasa res:c002 .

res:p003 a hog:Profesor ;
    hog:nombre "juan pablo" ;
    hog:perteneceACasa res:c003 .




## 8. Verificación final

Antes de terminar, compruebe que:

- todas las instancias esperadas fueron creadas
- los IRI siguen el esquema diseñado en el ejercicio anterior
- los literales tienen el tipo de dato apropiado cuando corresponde
- las relaciones entre recursos utilizan IRI y no cadenas de texto
- el Turtle generado es válido
- el grafo contiene las relaciones previstas por su modelo RDFS


# Punto 6: Preguntas de analisis

6.1. Un vocabulario declara que ex:perteneceACasa tiene como dominio ex:Estudiante y como
rango ex:Casa. En el grafo aparece la siguiente tripleta:
ex:Dumbledore ex:perteneceACasa ex:Gryffindor .
Suponga que ex:Dumbledore no había sido declarado previamente como ex:Estudiante.
¿Considera RDFS que la tripleta es inválida? ¿Qué puede inferirse a partir de ella? Explique.

<br>
No es invalida, porque el validador detecta que el dominio es ex:Estudiante, quiere decir que sin importar lo que se ponga como sujeto y sin importar que se haya declarado previamente para ese sujeto, el programa va a inferir que es de tipo Estudiante, porque el dominio del predicado esta establecido.

6.2. Después de calcular la clausura RDFS de un grafo, se observa que contiene la tripleta:
ex:Harry rdf:type ex:Persona .
Sin embargo, esa tripleta no aparecía en los datos originales. Explique cómo puede formar
parte del grafo resultante sin haber sido afirmada explícitamente. Proponga dos caminos
distintos mediante los cuales RDFS podría haberla derivado.

<br>
Primer camino:
Puede que Harry se haya instanciado como un Estudiante, y previamente se definio que Estudiante es una subclase de Persona. Por lo tanto, el validador infiere que Harry es una Persona.

<br>
Segundo camino:
La otra posibilidad es por la definicion del rango de una propiedad, por ejemplo:<br>
:hechizoRealizadoPor  rdf:type rdf: Property ;
                      rdfs:domain ex: Hechizo ;
                      rdfs:range ex: Persona .

Entonces si existe alguna tripleta donde el objeto sea Harry y el predicado sea hechizoRealizadoPor, entonces el validador va a inferir que Harry es una Persona.


6.3. Un estudiante propone las siguientes relaciones para ampliar el modelo:
ex:Gryffindor rdfs:subClassOf ex:Casa .
ex:Profesor rdfs:subClassOf ex:Persona .
ex:Varita rdfs:subClassOf ex:ObjetoMagico .
¿Son correctas las tres? Para cada una, justifique su respuesta utilizando el significado de
rdfs:subClassOf y, si no es correcta, indique cómo debería representarse la relación.

<br>
La primera no es correcta, porque Gryffindor deberia ser una instancia de la clase Casa, no deberia ser subclase de Casa. Deberia representarse asi:
ex:Gryffindor rdf:type ex:Casa .<br>

La segunda es correcta, porque un profesor es una Persona, entonces su clase padre puede ser algo mas general como Persona, porque todo profesor es una Persona. <br>

La tercera también es correcta, siempre que Varita represente una clase. Una varita es un tipo de objeto mágico, por lo que Varita puede ser una subclase de ObjetoMagico.